# 2 · track_step — 이력을 보고 지금을 잇는다

t-4..t-1 의 검출 결과를 주고 t 의 물체를 답하게 합니다. 물체는 이미 가진 id 를 그대로 써야 하고, 처음 보는 것에만 새 id 를 줍니다. 표준적인 tracking 정식화입니다.

이 노트북은 네 가지를 확인합니다 — **어떤 원시 데이터에서**, **어떤 코드를 거쳐**, **무엇이 입력으로 들어가고**, **빌드된 파일이 그 코드와 일치하는지**. GPU 는 필요 없습니다.

In [ ]:
import os, sys, json, textwrap
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import numpy as np
import pandas as pd

from datatools import paths

ITEMS = os.path.join(paths.COMMON_DIR, "instruct_items_tasks01_06.parquet")
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 80)
wrap = lambda s, i="   ": textwrap.fill(str(s), 94, initial_indent=i,
                                        subsequent_indent=i)
VARIANTS = ['track_step_azdeg', 'track_step_bbox']
print("variants:", VARIANTS)

## 1. 어떤 원시 데이터에서 오는가

| 아카이브 | 주기 | 읽는 것 |
|---|---|---|
| `obstacle.offline` | 10 Hz | track_id 가 정체성의 근거 |
| `egomotion` | 10 Hz | 월드 좌표 변환 |
| `radar` | 20 / 12.7 Hz | 5초 창을 4 Hz 로 20스캔 |
| `camera intrinsics` | 클립당 1 | bbox 형식과 CoT 의 가려짐 계산 |

## 2. 어떤 코드를 거치는가

실행 순서입니다.

| 함수 | 하는 일 |
|---|---|
| `local_ids(boxes)` | track_id 를 첫 등장 순으로 1,2,3... 로 재번호. 원본 id 는 임의의 큰 수이고, 이 태스크가 묻는 것은 번호가 아니라 일관성입니다 |
| `step_frame(boxes, t, ids, camera, scan)` | 한 순간의 물체 + 카메라 방위각 + 가려짐 비율 + 레이더 측정 |
| `occluded_fraction(box, nearer)` | 앞선 물체들이 이 박스를 덮는 면적 비율 |
| `step_text(objects, form)` | 이력과 답을 같은 형식으로 |
| `step_reason(now, previous, form)` | CoT 근거 |

전부 `datatools/frame_objects.py` 와 `datatools/geometry.py` 에 있습니다.

In [ ]:
import inspect
from datatools import frame_objects as F
for name in ['local_ids', 'step_frame', 'occluded_fraction', 'step_text', 'step_reason']:
    fn = getattr(F, name, None)
    if fn is None:
        from datatools import geometry as G
        fn = getattr(G, name, None)
    if fn is None or not callable(fn):
        print(f'{name}: (모듈 함수 아님)'); continue
    doc = (inspect.getdoc(fn) or '').split(chr(10))[0]
    print(f'{name:34s} {doc[:80]}')

## 3. 입력으로 무엇이 들어가는가

**비전 5장 (1 fps) · 레이더 20스캔 (5초, 4 Hz) · ego 20샘플 · 이력 4프레임**

입력 창은 태스크마다 다릅니다. 로더의 `WINDOWS` 표가 그것을 정하고, 레이더는 항상 20스캔이라 창의 길이가 바뀌면 샘플링 속도가 따라 바뀝니다 — 인코더 입력 모양은 변하지 않습니다.

In [ ]:
from training.instruct_data import WINDOWS, INSTANT_TASKS, WINDOW_TASKS
for v in VARIANTS:
    for name in (v, v + "_cot"):
        if name in WINDOWS:
            secs, hz, frames = WINDOWS[name]
            print(f"{name:26s} 창 {secs}초 · 레이더 {hz} Hz × 20스캔 · 비전 {frames}장")
        elif name in INSTANT_TASKS:
            print(f"{name:26s} 순간 — 비전 1장 · 레이더 20스캔/1초 · ego 1")
        else:
            print(f"{name:26s} 클립 전체 — 비전 20장 · 레이더 20스캔/20초")

## 4. 실제 아이템

빌드된 파일에서 그대로 꺼냅니다.

In [ ]:
built = pd.read_parquet(ITEMS)
for v in VARIANTS:
    sub = built[built.task == v]
    if sub.empty:
        print(f"{v}: 파일에 없음"); continue
    r = sub.iloc[0]
    print("=" * 96)
    print(f"{v}   clip {r.clip_id[:8]}  frame {r.frame} (t={r.frame-1}s)  split {r.split}")
    print("Q:"); print(wrap(r.prompt))
    print("A:"); print(wrap(r.target))

## 5. CoT — 근거가 답을 만드는가

`_cot` 변형은 `{"rationale": ..., "answer": ...}` 입니다. **근거를 따라가면 답이 나와야** 합니다. 나오지 않으면 그 사슬은 잘못된 것이고, 보상을 걸면 모델이 그 잘못된 사슬을 배웁니다.

In [ ]:
for v in VARIANTS:
    name = v + "_cot"
    sub = built[built.task == name] if 'built' in dir() else None
    if sub is None or sub.empty:
        continue
    r = sub.iloc[0]
    d = json.loads(r.target)
    print("=" * 96); print(name)
    print("R:"); print(wrap(d["rationale"]))
    print("A:"); print(wrap(d["answer"]))

## 6. 보상

평가 채점기에서 유도했습니다. 정답을 그대로 넣으면 1.0 이 나와야 하고, 내용을 망가뜨리면 떨어져야 합니다.

In [ ]:
import re
from training.task_scorers import reward_for

def wreck(text):
    """형식은 두고 숫자만 2배로."""
    return re.sub(r"\d+(?:\.\d+)?",
                  lambda m: str(round(float(m.group()) * 2, 1)), text)

rows = []
for v in VARIANTS:
    for name in (v, v + "_cot"):
        fn = reward_for(name)
        if fn is None:
            continue
        sub = built[built.task == name] if 'built' in dir() else None
        if sub is None or sub.empty:
            continue
        t = sub.iloc[0].target
        rows.append({"task": name, "reward": fn.__name__,
                     "정답": round(fn(t, t), 3),
                     "숫자 2배": round(fn(wreck(t), t), 3)})
pd.DataFrame(rows)

## 7. 데이터 양

`val` 은 `train` 에 합쳐져 있습니다 — 클립 분할이 train 86,607 / val 54,163 / test 37,121 인데, 모델 선택은 `test` 에서 하므로 검증용 3분의 1이 쓰이지 않고 있었습니다.

In [ ]:
from collections import Counter
from training.instruct_data import load_items
names = [v for v in VARIANTS] + [v + "_cot" for v in VARIANTS]
rows = []
for split in ("train", "test"):
    c = Counter(i["task"] for i in load_items(tuple(names), split))
    for n in names:
        rows.append({"task": n, "split": split, "items": c.get(n, 0)})
pd.DataFrame(rows).pivot(index="task", columns="split", values="items")

## 8. 이 태스크에서 내린 결정과 근거

**앵커가 1초 간격인 이유**

롤아웃에서 이력은 자기 출력이므로 앵커 간격이 곧 이력 간격이 됩니다. 3초 간격 앵커에 1초 간격 이력을 쓰면 학습과 평가의 프롬프트가 달라집니다.

**id 는 값이 아니라 일관성으로 채점**

첫 차를 #2 라 불러도 계속 #2 이면 맞습니다. 항목 단위로는 프롬프트의 이력이 정한 id 를 이어받았는지 보고, 시퀀스 단위로는 IDF1 로 대응을 한 번 풀어 셉니다.

**가려짐이 76% 항목에 등장**

물체의 25.6% 가 이미지에서 절반 이상 가려지고, 그중 48.2% 는 레이더가 여전히 봅니다. 융합이 값을 하는 지점입니다.